# Notebook 02: Control de Calidad, Limpieza y Escalamiento de Datos

**Objetivo:** Aplicar las reglas de gobernanza sobre el dataset real de Kaggle: tratar valores nulos, eliminar outliers, realizar la división de datos en Train/Test y escalar las variables cuantitativas para garantizar la correcta convergencia de los pesos en la red neuronal.

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Carga resiliente (Lee de la sesión local o descarga de GitHub)
local_path = 'data/raw/credit_risk_dataset.csv'
url_dataset = "https://raw.githubusercontent.com/pr1m8/Credit-Default/main/credit_risk_dataset.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print("Dataset cargado desde la carpeta local data/raw.")
else:
    df = pd.read_csv(url_dataset)
    cols_interes = [
        'person_age', 'person_income', 'loan_amnt',
        'loan_int_rate', 'loan_percent_income',
        'cb_person_cred_hist_length', 'loan_status'
    ]
    df = df[cols_interes].copy()
    print("Dataset cargado desde la fuente remota.")

print(f"Dimensiones iniciales: {df.shape[0]} filas x {df.shape[1]} columnas.")

Dataset cargado desde la fuente remota.
Dimensiones iniciales: 32581 filas x 7 columnas.


In [6]:
# 1. Filtro de Outliers: Eliminar registros con edades biológicamente imposibles
df_clean = df[df['person_age'] <= 100].copy()
registros_eliminados = len(df) - len(df_clean)
print(f"Registros eliminados por edad atípica (>100 años): {registros_eliminados}")

# 2. Imputación de Nulos sin usar inplace=True (Asignación directa limpia)
mediana_tasa = df_clean['loan_int_rate'].median()
df_clean['loan_int_rate'] = df_clean['loan_int_rate'].fillna(mediana_tasa)
print(f"Nulos imputados en 'loan_int_rate' con la mediana ({mediana_tasa}%).")

# Verificación final de nulos
print("\n--- Verificación de valores faltantes tras la limpieza ---")
print(df_clean.isnull().sum())

Registros eliminados por edad atípica (>100 años): 5
Nulos imputados en 'loan_int_rate' con la mediana (10.99%).

--- Verificación de valores faltantes tras la limpieza ---
person_age                    0
person_income                 0
loan_amnt                     0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
loan_status                   0
dtype: int64


In [7]:
# 1. Separar matriz de características (X) y vector target (y)
X = df_clean.drop(columns=['loan_status'])
y = df_clean['loan_status']

# 2. División estricta Train (80%) y Test (20%) manteniendo proporción de clases (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Conjunto de Entrenamiento (Train): {X_train.shape[0]} muestras")
print(f"Conjunto de Evaluación (Test): {X_test.shape[0]} muestras")

# 3. Escalamiento MinMaxScaler [0, 1] (Ajustado SOLO en Train para evitar Data Leakage)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir de nuevo a DataFrames estructurados
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\n--- Muestra del conjunto X_train transformado y escalado en [0, 1] ---")
display(X_train_scaled.head())

Conjunto de Entrenamiento (Train): 26060 muestras
Conjunto de Evaluación (Test): 6516 muestras

--- Muestra del conjunto X_train transformado y escalado en [0, 1] ---


,person_age,person_income,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
0,0.121622,0.029964,0.263768,0.021688,0.180723,0.107143
1,0.040541,0.012771,0.130435,0.512896,0.204819,0.035714
2,0.013514,0.041016,0.333333,0.326495,0.168675,0.000000
3,0.067568,0.034876,0.028986,0.065651,0.024096,0.071429
4,0.027027,0.005403,0.101449,0.273154,0.325301,0.000000


In [8]:
# Guardar los datos limpios en la carpeta local de la sesión de Colab
os.makedirs('data/processed', exist_ok=True)

X_train_scaled.to_csv('data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('data/processed/X_test_scaled.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print("Matrices procesadas y escaladas guardadas exitosamente en 'data/processed/'.")

Matrices procesadas y escaladas guardadas exitosamente en 'data/processed/'.
